# Backend Test: Stepwise American Option Comparative Workflow Agent

This notebook tests the **Recipe Workbench** backend end-to-end without the web UI, walking through the interactive dependency-discovery and comparative workflow pipeline:

1. **Pass `American Option`** (`Compare` mode) $\rightarrow$ agent returns downward dependency base classes (`pricing_model`, `volatility_model`, `pricing_engine`, `market_data`, `contract_spec`).
2. **Pass `models`** (and inspect `volatility_model` / `calculators`) $\rightarrow$ agent lists available ORE/QuantLib implementations for an American Option.
3. **Configure `N = 2` workflows** $\rightarrow$ agent partitions dependencies into **Shared (select 1)** vs. **Varied Comparison Axis (select 2)**.
4. **Select shared dependencies** $\rightarrow$ **SPY American Call** for today's date, **Black–Scholes–Merton** base process, **FD PDE** calculator, and **current 10-Year Treasury rate (`FRED:DGS10`)**.
5. **Select 2 volatility models** $\rightarrow$ **Workflow 1: `flat_vol`** (`FlatBlackScholesVol`) vs. **Workflow 2: `local_vol`** (`LocalVolSurface`).
6. **Execute in ORE-SWIG** $\rightarrow$ compare NPV, Delta, Chain RMSE, winner ranking, and a strike sweep across SPY Calls.

In [ ]:
import os
import sys
from datetime import date
from pathlib import Path

import pandas as pd

# Ensure repo root is working directory and src/ is importable
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO_ROOT)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from recipe_workbench.agent import ComparativeWorkflowAgent

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 140)

agent = ComparativeWorkflowAgent()
print(f"Initialized ComparativeWorkflowAgent | Graph version: {agent.graph.graph_version}")

## Step 1: Pass `American Option` to the Agent (Comparative Analysis)

The agent resolves `"American Option"` to the `EquityOption` root node in `OREDomainGraph` and returns the **downward dependency base classes** required to price and compare it.

In [ ]:
discovery = agent.start("American Option", mode="Compare")

print(f"Resolved Root : {discovery.root_id} ({discovery.root_label})")
print(f"Mode          : {discovery.mode}")
print(f"Next Prompt   : {discovery.next_prompt}\n")

deps_df = pd.DataFrame([c.model_dump() for c in discovery.dependency_categories])
deps_df[["category", "label", "base_class", "available_count", "can_vary_across_workflows"]]

## Step 2: User Passes `models` → Inspect Available Models & Volatility Models

Passing `"models"` returns the base stochastic process models available for an American Option (`BlackScholesMerton`, `Heston`). We also inspect `"volatility_model"` and `"calculators"` to see the concrete ORE options.

In [ ]:
models_resp = agent.inspect_dependency("models")
print(f"Category: {models_resp.category} | Base Class: {models_resp.base_class}")
print(f"Next Prompt: {models_resp.next_prompt}\n")
pd.DataFrame([o.model_dump() for o in models_resp.options])[["key", "label", "ore_class", "compatible_with"]]

In [ ]:
vol_resp = agent.inspect_dependency("volatility_model")
print(f"Category: {vol_resp.category} | Base Class: {vol_resp.base_class}")
vol_df = pd.DataFrame([o.model_dump() for o in vol_resp.options])[["key", "label", "compatible_with", "description"]]
display(vol_df)

calc_resp = agent.inspect_dependency("calculators")
print(f"\nCategory: {calc_resp.category} | Base Class: {calc_resp.base_class}")
calc_df = pd.DataFrame([o.model_dump() for o in calc_resp.options])[["key", "label", "description"]]
display(calc_df)

## Step 3: How Many Workflows? (`N = 2`) — Select 1 Shared Base Model vs. 2 Volatility Models

When comparing **2 workflows** along `volatility_model`, every other dependency slot (`pricing_model`, `pricing_engine`, `market_data`, `contract_spec`) takes **1 shared selection**, while `volatility_model` takes **2 selections** (one per workflow).

In [ ]:
plan = agent.configure_workflows(num_workflows=2, varied_category="volatility_model")

print(f"Workflows      : {plan.num_workflows}")
print(f"Varied Axis    : {plan.varied_category}")
print(f"Selection Rule : {plan.selection_rule}\n")

pd.DataFrame(
    [
        {
            "dependency_slot": slot,
            "role": "VARIED (per workflow)" if slot == plan.varied_category else "SHARED (fixed)",
            "selections_required": count,
        }
        for slot, count in plan.slot_counts.items()
    ]
)

## Step 4: Select Shared Dependencies (SPY Call, Today's Date, BSM, 10Y Treasury)

We bind the shared dependencies across both workflows:
- **Contract**: `SPY` American **Call**, valuation date = today, ~30 DTE ATM
- **Base Model**: `BlackScholesMerton`
- **Calculator**: `fd_bs` (`ORE.FdBlackScholesVanillaEngine`)
- **Interest Rate**: Current **10-Year Treasury (`10Y` $\rightarrow$ `FRED:DGS10`)**

In [ ]:
today_iso = date.today().isoformat()

shared = agent.select_shared(
    pricing_model="BlackScholesMerton",
    pricing_engine="fd_bs",
    symbol="SPY",
    option_type="Call",
    exercise_style="American",
    asof=today_iso,
    target_dte=30,
    moneyness="ATM",
    rate_tenor="10Y",
)

pd.DataFrame([shared.model_dump()]).T.rename(columns={0: "Selected Value"})

## Step 5: Select 2 Volatility Models (`flat_vol` vs. `local_vol`)

Both `flat_vol` (`FlatBlackScholesVol`) and `local_vol` (`LocalVolSurface`) are compatible with `BlackScholesMerton`. The agent constructs 2 explicit workflow specs.

In [ ]:
workflows = agent.select_varied(["flat_vol", "local_vol"])
pd.DataFrame([w.model_dump() for w in workflows])

## Step 6: Execute Both Workflows in ORE-SWIG

This fetches today's SPY option chain + live `FRED:DGS10` 10-Year Treasury rate when online (and falls back cleanly to the immutable SPY gold snapshot + 10Y Treasury fallback when offline), then prices and ranks both workflows in ORE.

In [ ]:
report = agent.execute(allow_network=True, run_strike_sweep=True)

print("=== Execution Context ===")
print(f"Requested As-Of : {report.asof_requested} | Effective As-Of: {report.asof_effective}")
print(f"Snapshot        : {report.snapshot_id}")
print(f"Spot (SPY)      : ${report.spot:.2f}")
print(f"10Y Risk-Free r : {report.risk_free_rate:.4%}  [{report.rate_source}]")
print(f"Dividend Yield  : {report.dividend_yield:.4%}")
print(
    f"Contract        : {report.contract.contract_symbol} "
    f"({report.contract.exercise_style} {report.contract.option_type}, "
    f"K=${report.contract.strike:.2f}, Exp={report.contract.expiry}, DTE={report.contract.dte})"
)
print(f"Winner (RMSE)   : {report.comparison.winner_model_name} ({report.comparison.winner_model_id})")
for note in report.notes:
    print(f"  - {note}")

comp_df = pd.DataFrame([r.model_dump() for r in report.comparison.rows])
comp_df[["rank", "model_id", "model_name", "engine_name", "price", "delta", "chain_rmse", "elapsed_ms", "notes"]]

## Step 7: Strike Sweep Comparison & Chart (OTM $\rightarrow$ ATM $\rightarrow$ ITM SPY Calls)

Comparing `flat_vol` vs. `local_vol` across strikes around spot shows where skew/curvature causes the two volatility models to diverge.

In [ ]:
sweep_records = []
for row in report.strike_sweep:
    rec = {
        "strike": row.strike,
        "moneyness_%": row.moneyness_pct,
        "market_mid": row.market_mid,
        "market_iv_%": row.market_iv,
    }
    for k, v in row.workflow_prices.items():
        rec[f"{k}_price"] = v
    for k, v in row.workflow_deltas.items():
        rec[f"{k}_delta"] = v
    rec["local_minus_flat"] = row.price_diff
    sweep_records.append(rec)

sweep_df = pd.DataFrame(sweep_records)
sweep_df

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # Left: Model prices vs Market Mid across strikes
    axes[0].plot(sweep_df["strike"], sweep_df["market_mid"], "ko-", label="Market Mid", linewidth=2)
    axes[0].plot(sweep_df["strike"], sweep_df["flat_vol_price"], "s--", label="Workflow 1: Flat Vol")
    axes[0].plot(sweep_df["strike"], sweep_df["local_vol_price"], "^-", label="Workflow 2: Local Vol")
    axes[0].axvline(report.spot, color="gray", linestyle=":", label=f"Spot (${report.spot:.2f})")
    axes[0].set_title(f"SPY American Call Prices ({report.contract.expiry}, r={report.risk_free_rate:.2%})")
    axes[0].set_xlabel("Strike ($)")
    axes[0].set_ylabel("Option Price ($)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Right: Model price spread (Local Vol - Flat Vol)
    colors = ["#2b8a3e" if x >= 0 else "#c92a2a" for x in sweep_df["local_minus_flat"]]
    axes[1].bar(sweep_df["strike"], sweep_df["local_minus_flat"], width=1.6, color=colors)
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_title("Price Spread: Local Vol − Flat Vol ($)")
    axes[1].set_xlabel("Strike ($)")
    axes[1].set_ylabel("Price Difference ($)")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib is not installed yet. Run `uv add --dev ipykernel matplotlib` to render charts.")
    print(sweep_df[["strike", "market_mid", "flat_vol_price", "local_vol_price", "local_minus_flat"]].to_string(index=False))